# 01_on_error — Explicit error handling

`@on_error` moves error handling out of the operation body into a separate declaration: "errors of this type are handled like this." The aspect just raises; a handler matched by `isinstance(error, type)` runs and **returns a `Result`** — which becomes the operation's output, so a meaningful answer comes out instead of an exception.

This notebook logs in a user; the summary step raises `ValueError` for a bad username, and an `@on_error(ValueError)` handler turns it into a `login_failed` result.

**What's new**

| Concept | Description |
|---------|-------------|
| `@on_error(ExcType, description=...)` | Declares a handler for an exception type |
| handler signature | `(self, params, state, box, connections, error)` |
| return `Result` | Closes the error — the operation finishes normally |
| matching | First handler whose type matches `isinstance` (declare specific → general) |

In [ ]:
!pip install aoa-action-machine

In [ ]:
import asyncio

from pydantic import Field

from aoa.action_machine.auth import NoneRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.intents.on_error import on_error
from aoa.action_machine.logging import Channel, ConsoleLogger
from aoa.action_machine.logging.log_coordinator import LogCoordinator
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

## Domain, Params, Result

In [ ]:
class RootDomain(BaseDomain):
    name = "root"
    description = "Root domain"


class LoginParams(BaseParams):
    username: str = Field(description="Username")


class LoginResult(BaseResult):
    username: str = Field(description="Username")
    status: str = Field(description="Status")

## The action — summary raises, `@on_error` catches

`login_summary` raises `ValueError` for the username `"bad"`. The `@on_error(ValueError)` handler catches it, logs a business event, and returns a `LoginResult` with `status="login_failed"` — so the operation completes with a real result, not an exception.

In [ ]:
@meta(description="Login", domain=RootDomain)
@check_roles(NoneRole)
class LoginAction(BaseAction[LoginParams, LoginResult]):

    @summary_aspect("Validate credentials")
    async def login_summary(self, params, state, box, connections):
        if params.username == "bad":
            raise ValueError("invalid credentials")
        return LoginResult(username=params.username, status="ok")

    @on_error(ValueError, description="Invalid credentials")
    async def validation_error_on_error(self, params, state, box, connections, error):
        await box.info(Channel.business, "on_error: {%var.msg}", msg=str(error))
        return LoginResult(username=params.username, status="login_failed")

## Run

The summary fails with `ValueError`; the handler intercepts it, records the event, and returns the failed-login result.

> In Colab, `await` works at top level — no `asyncio.run()`.

In [ ]:
async def main() -> None:
    machine = ActionProductMachine(
        log_coordinator=LogCoordinator(loggers=[ConsoleLogger()])
    )
    result = await machine.run(
        Context(),
        LoginAction(),
        params=LoginParams(username="bad"),
    )
    print("\n" + f"Result: username={result.username}, status={result.status}")


await main()